# Donut EDS Adapter

This notebook processes PDF documents using Donut (Document Understanding Transformer) as an open-source alternative to AWS Textract. It:
1. Reads the zero-shot results CSV to identify documents with forms
2. Extracts the lowest page number from each document's form_pages
3. Sends individual pages to Donut for structured information extraction

In [2]:
import pandas as pd
import os
from pathlib import Path
import json
from typing import List, Dict, Any
import PyPDF2
from io import BytesIO
import base64
from PIL import Image
import fitz  # PyMuPDF
import torch
from transformers import DonutProcessor, VisionEncoderDecoderModel
import re
from datetime import datetime

## Configuration

In [ ]:
# Configuration
CLOBBER = False  # Set to True to overwrite existing results, False to skip already processed files

# Agency filtering toggle - set to None to process all agencies, or specify agency name(s) to filter
# Examples:
# FILTER_AGENCIES = None  # Process all agencies
# FILTER_AGENCIES = "Education"  # Process only Education agency
# FILTER_AGENCIES = ["Education", "Correction"]  # Process multiple agencies
FILTER_AGENCIES = None #"Correction"

# File paths
CSV_PATH = "../../preprocessing/zero_shot_results_full_corpus.csv"
CONTRACTS_DIR = "../../../data/raw/_contracts/"
OUTPUT_DIR = "../../../data/intermediate_products/eds_forms_donut/"

# Donut configuration
MODEL_NAME = "naver-clova-ix/donut-base-finetuned-docvqa"  # Pre-trained DocVQA model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LENGTH = 512  # Maximum token length for generation
IMAGE_SIZE = [1280, 960]  # Image size for processing

# Queries for the EDS form extraction - updated section 3 and section 13 queries
DEFAULT_QUERIES = [
    "EDS Number",
    "Date Prepared",
    "Is 'Professional/Personal Services' checked in section 3?",
    "Is 'Grant' checked in section 3?",
    "Is 'Lease' checked in section 3?",
    "Is 'Attorney' checked in section 3?",
    "Is 'MOU' checked in section 3?",
    "Is 'QPA' checked in section 3? If yes, what is written after it?",
    "Is 'Contract for procured Services' checked in section 3?",
    "Is 'Maintenance' checked in section 3?",
    "Is 'License Agreement' checked in section 3?",
    "Is 'Amendment #' checked in section 3? If yes, what number?",
    "Is 'Renewal #' checked in section 3? If yes, what number?",
    "Is 'Other' checked in section 3? If yes, what is written after it?",
    "Total amount this action:",
    "New contract total",
    "From (month/day, year)",
    "To (month, day, year)",
    "Is 'Bid/Quotation' selected in section 13 Method of source selection?",
    "Is 'Emergency' selected in section 13 Method of source selection?",
    "Is 'Negotiated' selected in section 13 Method of source selection?",
    "Is 'Special Procurement' selected in section 13 Method of source selection?",
    "Is 'RFP #' selected in section 13 Method of source selection? If yes, what number?",
    "Is 'Other (specify)' selected in section 13 Method of source selection? If yes, what is specified?",
    "Name of agency:",
    "Vendor ID #",
    "Vendor Name",
    "Is \"Yes\" checked for Primary Vendor: Minority:'?",
    "Is \"Yes\" checked for Primary Vendor: Women:'?",
    "Is \"Yes\" checked for 'Is there Renewal Language in the document'?",
    "Is \"Yes\" checked for 'Is there a Termination for Convenience\" clause in the document'?"
]

print(f"Using device: {DEVICE}")
print(f"Loading Donut model: {MODEL_NAME}")

## Load Donut Model

In [6]:
# Load Donut processor and model
print("Loading Donut processor and model...")
processor = DonutProcessor.from_pretrained(MODEL_NAME,use_fast=True)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

if DEVICE == "cuda":
    model.half()  # Use half precision for faster inference on GPU
    
model.to(DEVICE)
model.eval()

print(f"Model loaded successfully on {DEVICE}")

Loading Donut processor and model...
Model loaded successfully on cuda


## Load and Filter Data

In [ ]:
# Load professional services contracts data
with open("../../../data/raw/indiana_prof_services_contracts.json", 'r') as f:
    prof_services_contracts = json.load(f)

print(f"Total professional services contracts: {len(prof_services_contracts)}")
[contract['agencyName'] for contract in prof_services_contracts].count('Correction')
#tabulate([contract['agencyName'] for contract in prof_services_contracts])

In [10]:
# Apply agency filter if specified
if FILTER_AGENCIES is not None:
    print(f"\n=== AGENCY FILTER APPLIED ===")
    
    # Handle both single agency string and list of agencies
    if isinstance(FILTER_AGENCIES, str):
        target_agencies = [FILTER_AGENCIES]
    else:
        target_agencies = FILTER_AGENCIES
    
    print(f"Filtering for agencies: {target_agencies}")
    
    # Filter contracts by agency
    original_count = len(prof_services_contracts)
    prof_services_contracts = [
        contract for contract in prof_services_contracts 
        if contract['agencyName'] in target_agencies
    ]
    
    print(f"Contracts after agency filter: {len(prof_services_contracts)} (reduced from {original_count})")
    
    # Show agency distribution
    agency_counts = {}
    for contract in prof_services_contracts:
        agency = contract['agencyName']
        agency_counts[agency] = agency_counts.get(agency, 0) + 1
    
    print("Agency distribution:")
    for agency, count in sorted(agency_counts.items()):
        print(f"  {agency}: {count} contracts")
    print("=" * 30)
else:
    print("No agency filter applied - processing all agencies")

# Extract PDF filenames from professional services contracts
prof_services_filenames = set()
for contract in prof_services_contracts:
    pdf_url = contract['pdfUrl']
    # Extract filename from URL (e.g., "0000000000000000000011571-011.pdf")
    filename = pdf_url.split('/')[-1]
    prof_services_filenames.add(filename)

print(f"Unique professional services contract PDF files: {len(prof_services_filenames)}")

# Load the CSV file
df = pd.read_csv(CSV_PATH)
print(f"Total documents in CSV: {len(df)}")

# Filter for documents containing forms AND are professional services contracts
forms_df = df[df['contains_form'] == True].copy()
print(f"Documents with forms (before prof services filter): {len(forms_df)}")

# Filter to only include professional services contracts
prof_services_forms_df = forms_df[forms_df['filename'].isin(prof_services_filenames)].copy()
print(f"Professional services documents with forms: {len(prof_services_forms_df)}")

# Display sample of filtered data
print("\nSample of professional services documents with forms:")
print(prof_services_forms_df[['filename', 'form_pages', 'num_form_pages']].head())

# Update the working dataframe for the rest of the notebook
forms_df = prof_services_forms_df

No agency filter applied - processing all agencies
Unique professional services contract PDF files: 34243
Total documents in CSV: 160751
Documents with forms (before prof services filter): 68696
Professional services documents with forms: 26089

Sample of professional services documents with forms:
                                filename form_pages  num_form_pages
3996   0000000000000000000031808-000.pdf          1               1
9423   0000000000000000000040451-000.pdf      16,35               2
10225  0000000000000000000041271-001.pdf         17               1
11065  0000000000000000000048512-002.pdf          1               1
12541  0000000000000000000055177-000.pdf         16               1


/tmp/ipykernel_667444/1669605917.py:46: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_PATH)


## Helper Functions

In [ ]:
def get_lowest_page_number(form_pages_str: str) -> int:
    """
    Extract the lowest page number from the form_pages string.
    
    Args:
        form_pages_str: String containing page numbers (e.g., "1,2,4,10,15,16" or "5")
    
    Returns:
        int: The lowest page number
    """
    if pd.isna(form_pages_str) or form_pages_str == "":
        return None
    
    # Handle both single numbers and comma-separated lists
    if ',' in str(form_pages_str):
        page_numbers = [int(x.strip()) for x in str(form_pages_str).split(',')]
    else:
        page_numbers = [int(str(form_pages_str).strip())]
    
    return min(page_numbers)

def get_output_filename(filename: str, page_number: int) -> str:
    """
    Generate the output JSON filename for a given PDF and page number.
    
    Args:
        filename: Original PDF filename
        page_number: Page number
    
    Returns:
        str: Output JSON filename
    """
    filename_base = Path(filename).stem
    return f"{filename_base}_page_{page_number}_donut.json"

def file_already_processed(filename: str, page_number: int, output_dir: Path) -> bool:
    """
    Check if a file has already been processed.
    
    Args:
        filename: Original PDF filename
        page_number: Page number
        output_dir: Output directory path
    
    Returns:
        bool: True if file exists and has been processed
    """
    json_filename = get_output_filename(filename, page_number)
    json_path = output_dir / json_filename
    return json_path.exists() and json_path.stat().st_size > 0

def pdf_page_to_image(pdf_path: str, page_number: int, dpi: int = 200) -> Image.Image:
    """
    Convert a single PDF page to a PIL Image.
    
    Args:
        pdf_path: Path to the PDF file
        page_number: Page number to extract (1-indexed)
        dpi: Resolution for the image conversion
    
    Returns:
        PIL.Image: Image of the PDF page
    """
    doc = fitz.open(pdf_path)
    page = doc.load_page(page_number - 1)  # PyMuPDF uses 0-based indexing
    
    # Convert to image with specified DPI
    mat = fitz.Matrix(dpi / 72, dpi / 72)  # 72 is the default DPI
    pix = page.get_pixmap(matrix=mat)
    img_data = pix.tobytes("ppm")
    
    doc.close()
    
    return Image.open(BytesIO(img_data))

def process_with_donut(image: Image.Image, queries: List[str] = None) -> Dict[str, Any]:
    """
    Process an image with Donut using document VQA (Visual Question Answering).
    
    Args:
        image: PIL Image of the document page
        queries: List of queries to ask about the document
    
    Returns:
        dict: Dictionary containing query results and metadata
    """
    try:
        # Use provided queries or default queries
        if queries is None:
            queries = DEFAULT_QUERIES
        
        results = {
            'processing_timestamp': datetime.now().isoformat(),
            'model_name': MODEL_NAME,
            'device': DEVICE,
            'queries_and_answers': [],
            'raw_responses': []
        }
        
        for query in queries:
            try:
                # Prepare the task prompt for DocVQA
                task_prompt = f"<s_docvqa><s_question>{query}</s_question><s_answer>"
                
                # Process the image and text
                pixel_values = processor(image, return_tensors="pt").pixel_values
                
                if DEVICE == "cuda":
                    pixel_values = pixel_values.half()
                    
                pixel_values = pixel_values.to(DEVICE)
                
                # Encode the task prompt
                decoder_input_ids = processor.tokenizer(
                    task_prompt, 
                    add_special_tokens=False, 
                    return_tensors="pt"
                ).input_ids.to(DEVICE)
                
                # Generate answer - removed unsupported flags
                with torch.no_grad():
                    outputs = model.generate(
                        pixel_values,
                        decoder_input_ids=decoder_input_ids,
                        max_length=MAX_LENGTH,
                        pad_token_id=processor.tokenizer.pad_token_id,
                        eos_token_id=processor.tokenizer.eos_token_id,
                        use_cache=True,
                        num_beams=1,
                        bad_words_ids=[[processor.tokenizer.unk_token_id]],
                        return_dict_in_generate=True
                    )
                
                # Decode the generated sequence
                sequence = processor.batch_decode(outputs.sequences)[0]
                
                # Extract answer from the sequence
                answer = sequence.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "")
                
                # Parse answer from the generated sequence
                answer_match = re.search(r'<s_answer>(.*?)</s_answer>', answer)
                if answer_match:
                    parsed_answer = answer_match.group(1).strip()
                else:
                    # Fallback: extract everything after <s_answer>
                    answer_start = answer.find('<s_answer>') + len('<s_answer>')
                    parsed_answer = answer[answer_start:].strip()
                
                # Store the result
                qa_result = {
                    'query': query,
                    'answer': parsed_answer,
                    'confidence_score': None  # Donut doesn't provide confidence scores directly
                }
                
                results['queries_and_answers'].append(qa_result)
                results['raw_responses'].append({
                    'query': query,
                    'raw_sequence': sequence,
                    'full_answer': answer
                })
                
                print(f"  Q: {query}")
                print(f"  A: {parsed_answer}")
                
            except Exception as e:
                print(f"  Error processing query '{query}': {str(e)}")
                results['queries_and_answers'].append({
                    'query': query,
                    'answer': f"ERROR: {str(e)}",
                    'confidence_score': None
                })
        
        return results
        
    except Exception as e:
        print(f"Error processing with Donut: {str(e)}")
        return {
            'error': str(e),
            'processing_timestamp': datetime.now().isoformat(),
            'model_name': MODEL_NAME,
            'device': DEVICE
        }

In [12]:
# Add lowest page number to dataframe
forms_df['lowest_page'] = forms_df['form_pages'].apply(get_lowest_page_number)

# Remove rows where we couldn't determine the lowest page
forms_df = forms_df.dropna(subset=['lowest_page'])
forms_df['lowest_page'] = forms_df['lowest_page'].astype(int)

print(f"Documents with valid page numbers: {len(forms_df)}")
print("\nSample with lowest page numbers:")
print(forms_df[['filename', 'form_pages', 'lowest_page']].head(10))

# Check how many files remain unprocessed before creating output directory
print(f"\n=== PROCESSING QUEUE SUMMARY ===")

# Show agency filter status
if FILTER_AGENCIES is not None:
    if isinstance(FILTER_AGENCIES, str):
        print(f"🔍 AGENCY FILTER: {FILTER_AGENCIES}")
    else:
        print(f"🔍 AGENCY FILTER: {', '.join(FILTER_AGENCIES)}")
else:
    print("🔍 AGENCY FILTER: None (processing all agencies)")

print(f"Total professional services documents with forms queued: {len(forms_df)}")

# Create output directory if it doesn't exist to check existing files
temp_output_dir = Path(OUTPUT_DIR)
temp_output_dir.mkdir(parents=True, exist_ok=True)

# Count already processed files
already_processed_count = 0
for idx, row in forms_df.iterrows():
    filename = row['filename']
    lowest_page = row['lowest_page']
    if file_already_processed(filename, lowest_page, temp_output_dir):
        already_processed_count += 1

remaining_to_process = len(forms_df) - already_processed_count
print(f"Already processed: {already_processed_count}")
print(f"Remaining to process: {remaining_to_process}")
print(f"Processing progress: {(already_processed_count/len(forms_df)*100):.1f}% complete")
print("=" * 35)

Documents with valid page numbers: 26089

Sample with lowest page numbers:
                                filename                  form_pages  \
3996   0000000000000000000031808-000.pdf                           1   
9423   0000000000000000000040451-000.pdf                       16,35   
10225  0000000000000000000041271-001.pdf                          17   
11065  0000000000000000000048512-002.pdf                           1   
12541  0000000000000000000055177-000.pdf                          16   
12999  0000000000000000000041742-000.pdf  156,670,671,1176,1690,1691   
13200  0000000000000000000067712-000.pdf                          16   
14923  0000000000000000000081653-000.pdf                          16   
15147  0000000000000000000082482-000.pdf                          16   
15351  0000000000000000000083053-000.pdf                       21,27   

       lowest_page  
3996             1  
9423            16  
10225           17  
11065            1  
12541           16  
12999 

## Process Documents

In [ ]:
# Create output directory if it doesn't exist
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# Use the already processed count from the previous cell to filter dataframe
if not CLOBBER and already_processed_count > 0:
    print(f"CLOBBER = False: Filtering out {already_processed_count} already processed files...")
    # Filter out already processed files
    remaining_df = []
    for idx, row in forms_df.iterrows():
        filename = row['filename']
        lowest_page = row['lowest_page']
        if not file_already_processed(filename, lowest_page, output_dir):
            remaining_df.append(row)
    
    if remaining_df:
        forms_df = pd.DataFrame(remaining_df)
        print(f"Remaining files to process: {len(forms_df)}")
    else:
        print("All files already processed.")
        forms_df = pd.DataFrame()  # Empty dataframe
elif not CLOBBER:
    print("CLOBBER = False: No existing results found. Processing all files.")
else:
    print("CLOBBER = True: Processing all files (will overwrite existing results)")

# Process each document
results = []
errors = []
skipped = []

for idx, row in forms_df.iterrows():
    filename = row['filename']
    lowest_page = row['lowest_page']
    
    pdf_path = os.path.join(CONTRACTS_DIR, filename)
    
    # Check if file exists
    if not os.path.exists(pdf_path):
        error_msg = f"File not found: {filename}"
        print(error_msg)
        errors.append({'filename': filename, 'error': error_msg})
        continue
    
    try:
        print(f"Processing {filename}, page {lowest_page}...")
        
        # Convert PDF page to image
        page_image = pdf_page_to_image(pdf_path, lowest_page)
        
        # Process with Donut
        donut_response = process_with_donut(page_image)
        
        if donut_response and 'error' not in donut_response:
            # Save result immediately to avoid losing work
            json_filename = get_output_filename(filename, lowest_page)
            json_path = output_dir / json_filename
            
            with open(json_path, 'w') as f:
                json.dump(donut_response, f, indent=2, default=str)
            
            result = {
                'filename': filename,
                'page_number': lowest_page,
                'json_file': json_filename,
                'status': 'success',
                'num_queries': len(donut_response.get('queries_and_answers', []))
            }
            results.append(result)
            print(f"✓ Successfully processed {filename} → {json_filename}")
        else:
            error_msg = f"Donut processing failed for {filename}"
            if donut_response and 'error' in donut_response:
                error_msg += f": {donut_response['error']}"
            print(f"✗ {error_msg}")
            errors.append({'filename': filename, 'error': error_msg})
            
    except Exception as e:
        error_msg = f"Error processing {filename}: {str(e)}"
        print(f"✗ {error_msg}")
        errors.append({'filename': filename, 'error': error_msg})
    
    # Clear GPU cache if using CUDA
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print(f"\n=== PROCESSING COMPLETE ===")
print(f"Successfully processed: {len(results)} documents")
print(f"Errors: {len(errors)} documents")
if len(results) + len(errors) > 0:
    success_rate = len(results)/(len(results)+len(errors))*100
    print(f"Success rate: {success_rate:.1f}%")

CLOBBER = False: No existing results found. Processing all files.
Processing 0000000000000000000031808-000.pdf, page 1...


The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  Q: EDS Number
  A: a47-19-340-906
  Q: Date Prepared
  A: 7/11/2018


The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  Q: Which response options in the "3. CONTRACTS & LEASES" box are checked?
  A: contracts & leaves
  Q: Total amount this action:
  A: $2,073,678.95


...

The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  Q: Which response options in the "3. CONTRACTS & LEASES" box are checked?
  A: contracts & leases
  Q: Total amount this action:
  A: $90,000.00


The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  Q: New contract total
  A: 90,000.00
  Q: From (month/day, year)
  A: 8/4/2017


The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


## Save Results

In [ ]:
# Save summary and error files
if results or already_processed_count > 0:
    # Count existing files if CLOBBER was False
    existing_files = []
    if not CLOBBER:
        for json_file in output_dir.glob("*.json"):
            if json_file.name != "processing_summary.json":
                existing_files.append(json_file.name)
    
    # Save summary of all results (including existing ones)
    summary = {
        'processing_method': 'donut',
        'model_name': MODEL_NAME,
        'device': DEVICE,
        'clobber_mode': CLOBBER,
        'agency_filter': FILTER_AGENCIES,
        'newly_processed_count': len(results),
        'error_count': len(errors),
        'total_existing_files': len(existing_files) if not CLOBBER else 0,
        'processing_timestamp': datetime.now().isoformat(),
        'newly_processed_files': [{
            'filename': r['filename'],
            'page_number': r['page_number'],
            'json_file': r['json_file'],
            'num_queries': r.get('num_queries', 0)
        } for r in results]
    }
    
    with open(output_dir / 'processing_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"Summary saved: {output_dir / 'processing_summary.json'}")

# Save errors if any
if errors:
    errors_df = pd.DataFrame(errors)
    errors_df.to_csv(output_dir / 'processing_errors.csv', index=False)
    print(f"Errors saved: {output_dir / 'processing_errors.csv'}")

print(f"Results saved to: {output_dir}")

## Display Sample Results

In [ ]:
# Display sample extracted content from first successful result
if len(results) > 0:
    sample_result = results[0]
    print(f"\n=== SAMPLE EXTRACTION ===")
    print(f"File: {sample_result['filename']} (page {sample_result['page_number']})")
    print(f"Model: {MODEL_NAME}")
    
    # Load the JSON file to get Donut response
    json_path = output_dir / sample_result['json_file']
    with open(json_path, 'r') as f:
        donut_response = json.load(f)
    
    # Show query-answer pairs
    qa_pairs = donut_response.get('queries_and_answers', [])
    print(f"\nExtracted {len(qa_pairs)} query-answer pairs:")
    print("-" * 60)
    
    for i, qa in enumerate(qa_pairs[:10]):  # Show first 10
        print(f"{i+1:2d}. Q: {qa['query']}")
        print(f"    A: {qa['answer']}")
        print()
    
    if len(qa_pairs) > 10:
        print(f"... and {len(qa_pairs) - 10} more query-answer pairs")
        
else:
    print("\n=== NO SAMPLE AVAILABLE ===")
    print("No files were successfully processed.")

## Model Performance Notes

### Donut vs Textract Comparison

**Advantages of Donut:**
- ✅ **No API costs** - runs locally
- ✅ **Privacy** - no data sent to external services
- ✅ **Open source** - fully customizable
- ✅ **End-to-end** - designed for document understanding

**Considerations:**
- ⚠️ **Performance** - May need fine-tuning for best EDS form extraction
- ⚠️ **GPU Memory** - Requires significant GPU memory for large images
- ⚠️ **Processing Speed** - Slower than Textract API calls
- ⚠️ **Accuracy** - Custom Textract adapter may be more accurate for specific form types

**Recommendations:**
1. **For Production**: Fine-tune Donut on your specific EDS forms for better accuracy
2. **For Cost Optimization**: Use Donut for bulk processing where cost is a major factor
3. **For High Accuracy**: Keep Textract custom adapter for critical extractions
4. **Hybrid Approach**: Use Donut for initial processing, Textract for verification of important documents